In [11]:
# imports
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
# گرفتن داده
wines = pd.read_csv('winequality-red.csv', sep=';')
print(wines.shape)

# اینجا داده رو از موارد تکراری پاک کردیم.(تنها مشکل دیتا ست همین بود!)
wines_clean = wines.drop_duplicates()
print(wines_clean.shape)

In [ ]:
# می‌خواهیم با استفاده از ویژگی‌های شیمیایی شراب، کیفیت آن (quality) را پیش‌بینی کنیم.
# ستون(ویژگی) qualty=y
# مابقی ستون ها که برای پیش بینی لازمه(ویژگی)=x

# تعین داده
X = wines_clean.drop("quality", axis=1)
y = wines_clean["quality"]

In [ ]:
# جداسازی داده آموزسی با داده آزمایشی

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# X_train → ویژگی‌های داده‌های آموزشی
# X_test → ویژگی‌های داده‌های آزمایشی
# y_train → جواب‌های مربوط به داده‌های آموزشی
# y_test → جواب‌های واقعی داده‌های آزمایشی
# test_size=0.2 → ۲۰٪ داده برای Test
# random_state=42 → تقسیم تصادفی را قابل تکرار می‌کند.

# از این لحظه به بعد، هر چیزی که قرار است از داده یاد بگیرد، فقط باید روی Train انجام شود.

In [ ]:
# ساخت وآموزش مدل
# این مدل نیازی به ترنسفورمر ها ندارد و بدون خط لوله است!
# در اینجا مدل واقعاً رابطه‌ی ورودی و خروجی را یاد می‌گیرد.
model = RandomForestRegressor(random_state=42) # اینجا فقط یک نمونه از دل رو ساختیم و آموزش ندادیم.

# آموزش مدل
model.fit(X_train, y_train) # اینجا مدل رابطه ویژگی ها با کیفیت شراب و میفهمد و یاد میگیرد.

In [ ]:
# پیش بینی

y_pred = model.predict(X_test)

In [ ]:
# ارزیابی مدل
# برای اینکه بفهمیم مدلمون چقدر خوب پیشبینی کرده از این معیار ها استفاده میکنیم.

# MAE: مدل به طور متوسط چقدر از مقدار واقعی فاصله دارد؟
mae = mean_absolute_error(y_test, y_pred)

# MSE: این تابع خطا را به توان 2 میرساند تا واکنش به خطا بیشتر شود.
mse = mean_squared_error(y_test, y_pred)

# RMSE: جذر mse است تا مقیاس قابل تفسیر باشد
rmse = np.sqrt(mse)

# R²: مدل ما چه مقدار از تغییرات موجود در y را توضیح می‌دهد؟
# 1: بسیار خوب - 0: تقریبا بهتر از یک بیسلاین ساده نیست - 0>: میتواند از یک بیسلاین ساده بدتر باشد.
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)

In [ ]:
# Cross-Validation: تقسیم چند بار داده و آموزش چند بار مدل برای ارزیابی بهتر
# ایده: تقسیم داده آموزشی به چند قسمت و هر بار یک قسمت را برای تست و قسمت های دیگر را برای آموزش در نظر میگیریم.
#┌────┬────┬────┬────┬────┐                     |
#│ F1 │ F2 │ F3 │ F4 │ F5 │ Train: F2 F3 F4 F5  | Train: F1 F3 F4 F5
#└────┴────┴────┴────┴────┘ Test:  F1           | Test:  F2

scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5, # یعنی داده را به 5 فولد تقسیم و 5 بار آموزش بده
    scoring="neg_mean_absolute_error"
)

print(scores) # چون سنجش MAE با Sklearn برعکسه امتیاز ها منفی هستن و هر چه کوچک تر بهتر

# نکته: این فرایند برای مقایسه مدل ها و بررسی پایدارتر مدل را انجام میده و با روند تست فرق داره.
#         Train / Test
#          │        │
#          │        └─── Test Set
#          ▼
#    Cross-Validation

In [ ]:
# Transformers - Preprocessing
# برای بعضی مدل‌ها، این اختلاف مقیاس می‌تواند مشکل ایجاد کند؛بنابراین ممکن است بخواهیم ویژگی‌ها را استانداردسازی کنیم.
# در این نمونه از مدل KNN استفاده می کنیم که فاصله نمونه ها حساس است.
knn = KNeighborsRegressor(n_neighbors=5) # n_neighbors=5 یعنی برای پیش‌بینی یک نمونه، به ۵ نمونه نزدیک‌تر نگاه می‌کند.

# ساخت نمونه از اسکلر
scaler = StandardScaler()

# آموزش اسکلر برای یافتن پارامتر های مناسب استاندارد سازی دیتاست
scaler.fit(X_train) #  با فیت کردن اسکلر یاد میگیرد که چطور داده را تبدیل کند.

# اعمال اسکلر روی دیتاست برای استاندارد سازی (تبدیل داده)
X_train_scaled = scaler.transform(X_train) # با استفاده از پارامتر هایی که یادگرفتی دیتاست رو تبدیل کن
X_test_scaled = scaler.transform(X_test) # برای تست فیت اسکلر اعمال نمیشود و اسکلری که با داده آموزشی یاد گرفته روی تست هم اعمال میشه.

# حالا آموزش مدل
knn.fit(X_train_scaled, y_train)

# حالا پیش بینی
y_pred_knn = knn.predict(X_test_scaled)

In [ ]:
# Pipeline: خط لوله: خودکار سازی فرایند آموزش

# اینجا هر فرایند را ایجاد میکنی که برای ما استاندارد سازی دیتاست و آموزش با مدل knn
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsRegressor(n_neighbors=5))
])
# اسم ها دلخواه هسند!

# و حالا فقط فیت کردن
pipeline.fit(X_train, y_train)

# و پیش بینی
y_pred_pipeline = pipeline.predict(X_test)

#  Cross-Validation یکی از دلایل استفاده از خط لوله برای است

# تغیر پارامتر های هر مرحله(مثلا مدل)
pipeline.set_params(model__n_neighbors=10) # عبارت قبل __ نام مرحله و بعد __ پارامتر مورد نظر

#اما Pipeline فقط برای کوتاه‌تر کردن کد نیست؛ یکی از ارزش‌های مهمش این است که مراحل preprocessing و مدل را به‌صورت یک workflow واحد نگه می‌دارد و در Cross-Validation کمک می‌کند preprocessing به شکل درست داخل هر fold انجام شود.

In [ ]:
# Hyperparameter - فرا پارامتر ها
# فراپارامتر مقداری است که قبل از آموزش مدل توسط ما تعیین می‌شود و نحوه یا ساختار یادگیری مدل را کنترل می‌کند.
# پارامتر ها توسط خود مدل در حین عملیات یادگیری تعین میشود و از داده یاد میگیرد. اما فراپارامتر ها قبل از یادگیری توسط ما تعین میشوند.

Hyp = KNeighborsRegressor(n_neighbors=5)

# n_neighbors=5 => برای پیش‌بینی هر نمونه، به 5 نمونه نزدیک نگاه می‌کنم.
# weights=distance | uniform => تعین میکند که وزن همسایه ها یکسان باشند یا نزدیکتر ها بیشتر

# ما Hyperparameter را مستقیماً بر اساس Test Set انتخاب نمی‌کنیم؛ انتخاب و تنظیم را با Cross-Validation روی Training Set انجام می‌دهیم و Test Set را برای ارزیابی نهایی نگه می‌داریم.

In [ ]:
# GridSearchCV: جستجوی بهترین ترکیب فراپارمتر ها به صورت خودکار

# تعین فرا پارامتر هایی که میخواهیم تست شوند
param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11],
    "model__weights": ["uniform", "distance"]
}
# کلید فراپارامتر بستگی دارد که خط لوله داشته باشد یا نه. اگر داشت با این فرمت و اگر نداشت ساده مینویسیم.

# GridSearchCV ساخت
grid_search = GridSearchCV(
    pipeline, # چه مدلی ارزیابی شود
    param_grid, # چه پرامتر هایی تست شود
    cv=5, # برای هر ترکیب روی 5-fold Cross Validation استفاده کن.
    scoring="neg_mean_absolute_error", # معیار امتیاز
    n_jobs=-1 # یعنی تا حد امکان از تمام هسته‌های CPU برای اجرای موازی استفاده کن.
)

# حالا اجرا برای ارزیابی
grid_search.fit(X_train, y_train)

# چون refit=True به‌صورت پیش‌فرض فعال است، بعد از پیدا کردن بهترین Hyperparameterها، estimator با آن تنظیمات روی کل دادهٔ آموزشی دوباره fit می‌شود.
# پس میتوانیم مین را برای پیشبینی استفاده کنیم
y_pred = grid_search.predict(X_test)


# حال خروجی هایی که به ما میدهد تا تصمیم بگیریم
print(grid_search.best_params_) # بهترین ترکیب
print(grid_search.best_score_) # بهترین امتیاز
print(grid_search.best_estimator_) # بهترین مدل

#--------------------------------------
# حالا انتخاب فراپارامتر به صورت رندوم
# این مورد برای زمانی مناسبه که تعداد ترکیب ها خیلی زیاد باشند.

param_distributions = {
    "model__n_neighbors": range(1, 31),
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    n_iter=20, # تعداد ترکیب هایی که امتحان میشن
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=42 # می‌توانیم نتیجهٔ نمونه‌برداری را قابل تکرار کنیم.
)

In [ ]:
# زنجیره پروژه

#   Dataset
#      ↓
#  Cleaning
#      ↓
#    X / y
#      ↓
# Train / Test Split
#      ↓
#    Model
#      ↓
#  Evaluation
#      ↓
# Cross Validation
#      ↓
#  Transformer
#      ↓
#   Pipeline
#      ↓
# Hyperparameters
#      ↓
# Manual Tuning
#      ↓
# GridSearchCV
#      ↓
# RandomizedSearchCV

In [ ]:
# کد کامل پروژه با خط لوله و GridSearchCV

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


wines = pd.read_csv("winequality-red.csv", sep=";")

wines = wines.drop_duplicates()

X = wines.drop("quality", axis=1)
y = wines["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsRegressor())
])

param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15, 20],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

y_pred = grid_search.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Best parameters:", grid_search.best_params_)
print("Best CV MAE:", -grid_search.best_score_)
print("Test MAE:", mae)
print("Test MSE:", mse)
print("Test RMSE:", rmse)
print("Test R²:", r2)